# Chapter 9 &mdash; DFA, NFA and RE Are Equally Powerful

**Concept 6 of the Chapter 9 decomposition:** *DFA, NFA, and RE Are Equally Powerful*

The cycle closes: RE&rarr;NFA&rarr;DFA&rarr;minimal DFA&rarr;RE, so all three describe exactly the regular languages.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter9/Concept-Three-Models-Equally-Powerful/Concept-Three-Models-Equally-Powerful.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Chapter<N>/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]

from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
from jove.Def_RE2NFA     import *
from jove.Def_NFA2RE     import *
from jove.AnimateDFA     import *

import jove; print('Jove loaded from', list(jove.__path__)[0])
import jove.AnimateDFA as _a; print('animation toolbar:',
      'ready' if hasattr(_a.AnimateDFA, '_ipython_display_')
      else 'STALE -- restart the runtime, then re-run')

## 1. The idea


Every link of the chain now exists as code:

* **RE &rarr; NFA** &mdash; Thompson constructions (Chapter 8, Concept 2);
* **NFA &rarr; DFA** &mdash; subset construction (Chapter 7, Concept 8);
* **DFA &rarr; minimal DFA** &mdash; frames or Brzozowski (Chapter 6, Chapter 7);
* **DFA &rarr; NFA** &mdash; trivially, a DFA is an NFA;
* **NFA &rarr; RE** &mdash; state elimination (this chapter).

Compose them any way you like and the language is preserved. That is the content of
"**DFA = NFA = RE**": three notations, one class of languages, with mechanical
translations in both directions.

Convenience is what differs: REs are good for *specifying*, DFA for *deciding*, NFA
for *designing*.

## 2. Definitions

### The full cycle

In [ ]:
def cycle(r0):
    N  = re2nfa(r0)                 # RE  -> NFA
    D  = nfa2dfa(N)                 # NFA -> DFA
    Dm = min_dfa(D)                 # DFA -> minimal DFA
    N2 = dfa2nfa(Dm)                # DFA -> NFA  (trivial)
    _, _, r1 = del_gnfa_states(mk_gnfa(N2))   # NFA -> RE
    return N, D, Dm, r1

### Which notation is convenient for what

In [ ]:
ROLES = [("RE",  "specifying a pattern compactly"),
         ("NFA", "designing -- guessing is allowed"),
         ("DFA", "deciding -- one pass, constant memory"),
         ("min DFA", "comparing -- canonical, so iso_dfa is a proof")]

## 3. Tests

Start from an RE, go all the way round, and land on the same language.

In [ ]:
r0 = "(0+1)*01(0+1)*"
N, D, Dm, r1 = cycle(r0)
print("start RE : %s" % r0)
print("NFA      : %d states" % len(N["Q"]))
print("DFA      : %d states" % len(D["Q"]))
print("min DFA  : %d states" % len(Dm["Q"]))
print("end RE   : %s" % r1)
back = min_dfa(nfa2dfa(re2nfa(r1)))
print("\nisomorphic to where we started? ", iso_dfa(Dm, back))
assert iso_dfa(Dm, back)

The same for a batch of starting expressions.

In [ ]:
for r in ["(0+1)*1(0+1)(0+1)", "0*1*", "(01)*", "(0+1)*11", "1(0+1)*0"]:
    _, _, Dm, r1 = cycle(r)
    ok = iso_dfa(Dm, min_dfa(nfa2dfa(re2nfa(r1))))
    print("%-22s -> RE of length %4d, round-trips: %s" % (r, len(r1), ok))
    assert ok

Going the other way round &mdash; start from a DFA &mdash; closes too.

In [ ]:
D0 = md2mc('''DFA
IF : 0 -> Od
IF : 1 -> IF
Od : 0 -> IF
Od : 1 -> Od
''')
_, _, r = del_gnfa_states(mk_gnfa(dfa2nfa(D0)))
print("RE for 'even number of 0s' :", r)
assert iso_dfa(min_dfa(D0), min_dfa(nfa2dfa(re2nfa(r))))
print("back to an isomorphic minimal DFA.")

So the three notations are interchangeable &mdash; and each is good at something different.

In [ ]:
for name, use in ROLES:
    print("%-10s %s" % (name, use))

## 4. Animation

The canonical machine at the centre of the cycle.

In [ ]:
from jove.AnimateDFA import *
AnimateDFA(min_dfa(nfa2dfa(re2nfa('(0+1)*01(0+1)*'))), FuseEdges=True)

## 5. Exercises


1. Time each conversion on a 10-state NFA. Which dominates?
2. Which direction of the cycle can blow up exponentially? Both?
3. Give a language where the RE is short and the minimal DFA is huge, and vice versa.

In [ ]:
# Your work for the exercises above.

## 6. Where next

In [ ]:
# Previous / next, and a search box for all 245 concepts.
# Type a chapter (Chapter7, ch7) or words from a title (pumping, subset).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:  load_here('Chapter7/Concept-...')
from jove.Nav import nav, load_here
nav(here='Chapter9/Concept-Three-Models-Equally-Powerful')